# Ticket counts by time band: Kolmogorov-Smirnov test

**Two-Sample Kolmogorov-Smirnov Test** · a worked example from the StatNex teaching guide

**The story.** An online shop answers support tickets two ways: a chatbot (18 tickets) and a human agent (9 tickets). The support dashboard shows only how many tickets were resolved in each time band (0-10, 10-20, 20-40, 40-60 and 60-80 minutes), not the exact minutes.

**The question.** Can we still tell that the two teams' time patterns differ when we only have grouped counts?

**To run it**, choose **Runtime → Run all**. Colab first warns that this notebook was not written by Google; it reads none of your data, so choose **Run anyway**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `06_nonparametric/ks_two_sample__grouped.csv` (5 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Minutes,Chatbot,Human agent
0-10,14.0,0.0
10-20,0.0,9.0
20-40,1.0,0.0
40-60,2.0,0.0
60-80,1.0,0.0

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)


def stars(p):
    """* for P < 0.05, ** for P < 0.01."""
    return "**" if p < 0.01 else "*" if p < 0.05 else ""

## The data

In [ ]:
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
samples1 = ["Chatbot"]
samples2 = ["Human agent"]
# Each column once, in the order the pairs below use them.
columns = list(dict.fromkeys(name for a in samples1 for b in samples2 for name in (a, b)))
# Frequency data: each column holds one sample's counts in the same classes, from the
# lowest class up, so every cell has to be a number.
counts = data[columns].apply(pd.to_numeric, errors="coerce")
sizes = {name: int(counts[name].sum()) for name in columns}
print(pd.Series(sizes, name="n"))

## Two-sample Kolmogorov-Smirnov test

In [ ]:
# D is the largest gap between the two cumulative distributions: at some value, the
# share of sample 1 at or below it minus the share of sample 2. The P value is read
# from the limiting (large-sample) Kolmogorov distribution of
# KS = sqrt(n1 n2 / (n1 + n2)) D, scipy's stats.kstwobign. ks_2samp's own P (exact,
# or method="asymp") uses finite-sample forms and differs; in small samples the
# limiting P is conservative. It also ignores ties, so with heavily tied or grouped
# data treat P as approximate.
tests = []
for a in samples1:
    for b in samples2:
        n1, n2 = sizes[a], sizes[b]
        cumulative_a = counts[a].cumsum() / counts[a].sum()
        cumulative_b = counts[b].cumsum() / counts[b].sum()
        d = (cumulative_a - cumulative_b).abs().max()
        ks = d * np.sqrt(n1 * n2 / (n1 + n2))
        p = stats.kstwobign.sf(ks)
        tests.append([a, b, d, ks, p, stars(p)])

tests = pd.DataFrame(tests, columns=["sample 1", "sample 2", "D", "KS", "P", ""])
print()
print(tests.to_string(index=False))

## What the result says

The two-sample Kolmogorov-Smirnov test compares two whole distributions, the full patterns of the values. Here it has no exact minutes, only the counts of tickets in each time band: 18 chatbot tickets and 9 human agent tickets in 5 bands.

At the end of each band, the test compares the share of chatbot tickets solved so far with the share of human tickets solved so far. The maximum difference is 0.78. The statistic KS is 1.91 and the two-sided P value is 0.0014. The P value is the chance of a gap this large if both channels had the same pattern.

So the grouped counts are enough: the two patterns clearly differ. Grouping loses nothing here because a band edge falls where the two channels separate. With other bands, the gap could look smaller.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.